# 25: Transformer Architecture — Blueprint + From Scratch

## Why This Matters

**Transformers power almost every modern AI system you've heard of:**

| Model | Architecture | What it does |
|-------|-------------|--------------|
| ChatGPT, GPT-4 | Decoder-only Transformer | Text generation |
| BERT, RoBERTa | Encoder-only Transformer | Understanding / search |
| T5, BART | Full Encoder-Decoder Transformer | Translation, summarization |
| Claude (this conversation) | Transformer | Everything |

**This is a two-act lesson:**
1. **Act 1 — Blueprint**: see the shape of a Transformer and train a tiny classifier with PyTorch's built-in pieces.
2. **Act 2 — Build from Scratch**: assemble a full encoder–decoder Transformer from the components you wrote in lessons 23–24, train it on a copy task, and generate autoregressively.

### The Web Dev Analogy

The Transformer is like **microservices architecture**:
- **Encoder**: Process input (like REST API ingestion)
- **Decoder**: Generate output (like response generation)
- **Attention**: Services communicate directly — no sequential bottleneck
- **Parallel**: All positions processed simultaneously (not one-by-one like RNNs)
- **Scalable**: Add more layers/heads → larger model, same pattern

## What You'll Learn
- [ ] Explain the full transformer architecture (encoder + decoder blocks)
- [ ] Understand residual connections and layer normalization
- [ ] Train a small Transformer classifier end-to-end
- [ ] Implement a full encoder–decoder Transformer from scratch using Pre-LN
- [ ] Train on a copy task and generate autoregressively
- [ ] Compare Pre-LN vs Post-LN training stability

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 23**: Self-attention and multi-head attention | You already implemented `scaled_dot_product_attention` and `MultiHeadAttention` there — we reuse those ideas here |
| **Lesson 24**: Positional encoding | Added to input embeddings before feeding into transformer blocks; we reuse the `PositionalEncoding` module pattern |
| **Lesson 9**: Vanishing gradients | Residual connections solve this — gradients can skip layers via shortcuts |
| **Lesson 12**: PyTorch training loop | Same pattern: forward → loss → backward → step |

# Act 1 — The Blueprint

Before we write every line from scratch, let's zoom out and look at the shape of a Transformer, the three variants you'll meet in the wild, and the hyperparameters that define model size. Then we'll train a tiny classifier using PyTorch's built-in pieces so you see the whole architecture working end-to-end.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import math

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("Ready to build Transformers!")

## 1. Transformer Overview

**Architecture:**
```
Input → Encoder → Decoder → Output
```

**Key Components:**
1. **Encoder**: Processes input sequence
   - Multi-head self-attention
   - Feed-forward network
   - Layer normalization
   - Residual connections

2. **Decoder**: Generates output sequence
   - Masked self-attention (can't see future)
   - Cross-attention to encoder output
   - Feed-forward network
   - Layer normalization
   - Residual connections

In [ ]:
print("Transformer Architecture:")
print("=" * 70)
print("\nENCODER STACK (N layers):")
print("  Input Embedding + Positional Encoding")
print("  \u2193")
print("  [Multi-Head Self-Attention \u2192 Add & Norm]")
print("  \u2193")
print("  [Feed-Forward \u2192 Add & Norm]")
print("  \u2193")
print("  ... (repeat N times)")
print("  \u2193")
print("  Encoder Output")

print("\nDECODER STACK (N layers):")
print("  Output Embedding + Positional Encoding")
print("  \u2193")
print("  [Masked Multi-Head Self-Attention \u2192 Add & Norm]")
print("  \u2193")
print("  [Multi-Head Cross-Attention (to Encoder) \u2192 Add & Norm]")
print("  \u2193")
print("  [Feed-Forward \u2192 Add & Norm]")
print("  \u2193")
print("  ... (repeat N times)")
print("  \u2193")
print("  Linear + Softmax")
print("  \u2193")
print("  Output Probabilities")

`★ Insight ─────────────────────────────────────`

**Key Transformer innovations:**
1. **No recurrence**: Parallel processing (faster training)
2. **Self-attention**: Direct connections between all positions
3. **Residual connections**: Enable very deep networks
4. **Layer norm**: Stable training
5. **Multi-head**: Multiple attention perspectives

These enable models with billions of parameters!

`─────────────────────────────────────────────────`

## 2. Model Variants

You'll meet three flavors of Transformers in the wild. They all use the same building blocks — they just keep different parts.

In [ ]:
print("Transformer Variants:")
print("=" * 70)

print("\n1. Encoder-Decoder (Original Transformer):")
print("   - Full encoder + decoder")
print("   - Use case: Translation, summarization")
print("   - Example: T5, BART")

print("\n2. Encoder-Only:")
print("   - Only encoder layers")
print("   - Bidirectional attention (sees all tokens)")
print("   - Use case: Classification, understanding")
print("   - Example: BERT, RoBERTa")

print("\n3. Decoder-Only:")
print("   - Only decoder layers (with causal masking)")
print("   - Autoregressive (left-to-right)")
print("   - Use case: Text generation, completion")
print("   - Example: GPT-2, GPT-3, GPT-4")

print("\n" + "=" * 70)
print("Modern trend: Decoder-only models dominate!")
print("  - Simpler architecture")
print("  - Scales to huge sizes")
print("  - Versatile (can do many tasks)")

## 3. Key Hyperparameters

In [ ]:
print("Transformer Hyperparameters:")
print("=" * 70)

configs = {
    "Tiny": {
        "d_model": 64,
        "num_heads": 4,
        "num_layers": 2,
        "d_ff": 256,
        "params": "~1M"
    },
    "Small (BERT-Small)": {
        "d_model": 512,
        "num_heads": 8,
        "num_layers": 6,
        "d_ff": 2048,
        "params": "~30M"
    },
    "Base (BERT-Base, GPT-2)": {
        "d_model": 768,
        "num_heads": 12,
        "num_layers": 12,
        "d_ff": 3072,
        "params": "~110M"
    },
    "Large (BERT-Large)": {
        "d_model": 1024,
        "num_heads": 16,
        "num_layers": 24,
        "d_ff": 4096,
        "params": "~340M"
    },
    "XL (GPT-3)": {
        "d_model": 12288,
        "num_heads": 96,
        "num_layers": 96,
        "d_ff": 49152,
        "params": "~175B"
    }
}

for name, config in configs.items():
    print(f"\n{name}:")
    for key, value in config.items():
        print(f"  {key}: {value}")

## 4. Let's Actually Train It — TinyClassifier

Building components is satisfying. Training them and watching the loss drop is better.

**Task**: sequence classification — given 5 random token IDs, predict if their average is above or below 10.

This is a real learning problem that requires the transformer to aggregate information across the sequence — exactly what attention is designed for. For Act 1 we use PyTorch's built-in `nn.TransformerEncoder` so we can focus on the shape of training. In Act 2 we'll swap it out for our own from-scratch implementation.

In [ ]:
# --- Setup: built-in Transformer encoder + classification head ---
class TinyClassifier(nn.Module):
    """PyTorch's built-in TransformerEncoder + a linear classification head."""

    def __init__(self, vocab_size=20, d_model=32, num_heads=4, num_layers=2, d_ff=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=d_ff,
            dropout=0.0, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, 2)  # 2 classes: low avg vs high avg

    def forward(self, x):
        embedded = self.embedding(x)         # (batch, seq_len, d_model)
        encoded = self.encoder(embedded)     # (batch, seq_len, d_model)
        pooled = encoded.mean(dim=1)         # mean pool -> (batch, d_model)
        return self.classifier(pooled)       # (batch, 2) logits

# Generate data
torch.manual_seed(42)
vocab_size, seq_len, n_samples = 20, 5, 1000
sequences = torch.randint(0, vocab_size, (n_samples, seq_len))
labels = (sequences.float().mean(dim=1) > vocab_size / 2).long()

n_train = 800
X_train_t, X_test_t = sequences[:n_train], sequences[n_train:]
y_train_t, y_test_t = labels[:n_train], labels[n_train:]

clf = TinyClassifier(num_heads=4)
n_params = sum(p.numel() for p in clf.parameters())
print(f"Task: predict if avg token ID > {vocab_size // 2}")
print(f"Model: 2 encoder layers, 4 heads, d_model=32 -> {n_params:,} parameters")
print(f"Class balance: {labels.float().mean():.1%} positive")
print(f"Random baseline accuracy: ~50%")

In [ ]:
# --- Train the Transformer ---
def train_tinyclassifier(model, n_epochs=25, batch_size=32, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    train_losses, train_accs, test_accs = [], [], []
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        for i in range(0, n_train, batch_size):
            xb = X_train_t[i:i+batch_size]
            yb = y_train_t[i:i+batch_size]
            loss = criterion(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        model.eval()
        with torch.no_grad():
            train_acc = (model(X_train_t).argmax(1) == y_train_t).float().mean().item()
            test_acc  = (model(X_test_t).argmax(1)  == y_test_t).float().mean().item()
        train_losses.append(epoch_loss)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:2d}: Loss={epoch_loss:.2f}  Train={train_acc:.1%}  Test={test_acc:.1%}")
    return train_losses, train_accs, test_accs

train_losses, train_accs, test_accs = train_tinyclassifier(clf)

# Save the 4-head accuracy for the num_heads ablation exercise later
acc_4head = test_accs[-1]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[1].plot(train_accs, label="Train")
axes[1].plot(test_accs, label="Test")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"\nYour Transformer learned! Final test accuracy: {acc_4head:.1%}")
print(f"   (Random baseline: ~50%)")
print(f"\n   This same pattern - encoder stack -> mean pooling -> linear head -")
print(f"   is exactly how BERT does sentence classification.")
print(f"   The only difference: BERT was pre-trained on 3.3B words first.")

# Act 2 — Build a Transformer From Scratch

Act 1 used PyTorch's built-in `nn.TransformerEncoder`. That's fine for a blueprint, but now you're going to build **every piece by hand** — multi-head attention, feed-forward, residual connections, layer norm — and assemble them into a full **encoder–decoder Transformer**, then train it to copy sequences and generate autoregressively.

You already implemented the core mechanics in earlier lessons:

- **Scaled dot-product attention and multi-head attention** — lesson 23
- **Sinusoidal positional encoding** — lesson 24

If any of those feel fuzzy, now is the moment to flip back and re-read the 30 seconds of intuition. Here we wire them together.

### Pre-LN vs Post-LN

There are two ways to arrange residual connections and layer norm:

- **Post-LN** (original 2017 paper): `x = LayerNorm(x + sublayer(x))`. Needs a learning-rate warmup to train stably.
- **Pre-LN** (what every modern model uses): `x = x + sublayer(LayerNorm(x))`. Trains stably without warmup.

**We're using Pre-LN** for the from-scratch implementation below. We'll revisit the comparison in the "What Can Go Wrong" section after training.

### Architecture (ASCII)

```
        Source                              Target (shifted right)
          |                                           |
     [Embed + PE]                               [Embed + PE]
          |                                           |
+---------v---------+                       +---------v---------+
| Encoder Block x N |                       | Decoder Block x N |
|  - Self-Attn      |---- cross-attn ------>|  - Masked Self-Attn
|  - FFN            |                       |  - Cross-Attn     |
|  (Pre-LN residual)|                       |  - FFN            |
+-------------------+                       +---------+---------+
                                                      |
                                                  [Linear]
                                                      |
                                               Output logits
```

## 5. Components (Pre-LN)

`PositionalEncoding`, `MultiHeadAttention`, and `FeedForward` are the exact pieces from lessons 23–24. We define them here so this notebook is self-contained and so we can wire them into a full encoder–decoder.

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding - same shape you built in lesson 24."""

    def __init__(self, d_model, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_seq_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Core attention op (lesson 23)."""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V), weights

class MultiHeadAttention(nn.Module):
    """Multi-head attention (lesson 23). Splits d_model across num_heads."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        attn, weights = scaled_dot_product_attention(Q, K, V, mask)
        attn = attn.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(attn), weights

class FeedForward(nn.Module):
    """Position-wise feed-forward: two linears with GELU in between."""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, x):
        return self.linear2(self.dropout(self.activation(self.linear1(x))))

def create_causal_mask(seq_len):
    """Lower-triangular mask so decoder can only look at past positions."""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

print("Components ready: PositionalEncoding, MultiHeadAttention, FeedForward, causal mask")

## 6. Encoder & Decoder Blocks (Pre-LN)

**Pre-LN pattern**: `x = x + sublayer(LayerNorm(x))` — normalize first, then apply the sub-layer, then add the residual. This is the pattern every modern Transformer (GPT-2/3/4, LLaMA, most of HuggingFace) actually uses.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Pre-LN encoder block: LayerNorm -> sublayer -> residual."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-LN: norm -> sublayer -> residual
        normed = self.norm1(x)
        attn_out, attn_weights = self.attention(normed, normed, normed, mask)
        x = x + self.dropout1(attn_out)

        normed = self.norm2(x)
        ff_out = self.ff(normed)
        x = x + self.dropout2(ff_out)
        return x, attn_weights

class TransformerDecoderBlock(nn.Module):
    """Pre-LN decoder block with masked self-attn, cross-attn, and FFN."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # 1) Masked self-attention
        normed = self.norm1(x)
        self_out, _ = self.self_attention(normed, normed, normed, tgt_mask)
        x = x + self.dropout1(self_out)
        # 2) Cross-attention to encoder output
        normed = self.norm2(x)
        cross_out, cross_weights = self.cross_attention(normed, encoder_output, encoder_output, src_mask)
        x = x + self.dropout2(cross_out)
        # 3) Feed-forward
        normed = self.norm3(x)
        ff_out = self.ff(normed)
        x = x + self.dropout3(ff_out)
        return x, cross_weights

print("Pre-LN encoder and decoder blocks defined.")

## 7. Full Transformer (Encoder + Decoder)

Stack the blocks, add embeddings and positional encoding at the input, and a linear projection at the output. That's a complete sequence-to-sequence Transformer.

In [ ]:
class Transformer(nn.Module):
    """Complete Pre-LN encoder-decoder Transformer for seq2seq tasks."""

    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=128,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=512,
        max_seq_len=512,
        dropout=0.1,
    ):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_encoder_layers)
        ])
        self.encoder_norm = nn.LayerNorm(d_model)
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_decoder_layers)
        ])
        self.decoder_norm = nn.LayerNorm(d_model)
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        self.d_model = d_model
        self.scale = math.sqrt(d_model)

    def encode(self, src, src_mask=None):
        x = self.src_embedding(src) * self.scale
        x = self.positional_encoding(x)
        for layer in self.encoder_layers:
            x, _ = layer(x, src_mask)
        return self.encoder_norm(x)

    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        x = self.tgt_embedding(tgt) * self.scale
        x = self.positional_encoding(x)
        for layer in self.decoder_layers:
            x, _ = layer(x, encoder_output, src_mask, tgt_mask)
        return self.decoder_norm(x)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc = self.encode(src, src_mask)
        dec = self.decode(tgt, enc, src_mask, tgt_mask)
        return self.output_projection(dec)

# Quick sanity check of the forward pass
model = Transformer(src_vocab_size=1000, tgt_vocab_size=1000)
src = torch.randint(0, 1000, (4, 20))
tgt = torch.randint(0, 1000, (4, 15))
tgt_mask = create_causal_mask(15)
with torch.no_grad():
    logits = model(src, tgt, tgt_mask=tgt_mask)
total_params = sum(p.numel() for p in model.parameters())
print(f"Full Transformer built from scratch - {total_params:,} params")
print(f"  src {tuple(src.shape)} + tgt {tuple(tgt.shape)} -> logits {tuple(logits.shape)}")

## 8. Training the Copy Task

The simplest seq2seq task: given an input sequence, output the same sequence. If our implementation is wrong somewhere, this task will expose it immediately.

Token conventions: `0 = PAD`, `1 = BOS`, `2 = EOS`, tokens 3+ are real data.

In [ ]:
def generate_copy_data(batch_size, seq_len, vocab_size=50):
    """Generate (src, tgt_input, tgt_output) for the copy task with teacher forcing."""
    sequences = torch.randint(3, vocab_size, (batch_size, seq_len))
    src = sequences
    bos = torch.ones(batch_size, 1, dtype=torch.long)  # BOS token = 1
    tgt_input = torch.cat([bos, sequences[:, :-1]], dim=1)
    tgt_output = sequences
    return src, tgt_input, tgt_output

# Peek at a sample
s_src, s_in, s_out = generate_copy_data(batch_size=2, seq_len=6)
print("Source (to copy):   ", s_src[0].tolist())
print("Target input (BOS+): ", s_in[0].tolist())
print("Target output:      ", s_out[0].tolist())

In [ ]:
vocab_size = 50
seq_len = 10
batch_size = 32
n_epochs = 100

copy_model = Transformer(
    src_vocab_size=vocab_size, tgt_vocab_size=vocab_size,
    d_model=64, num_heads=4,
    num_encoder_layers=2, num_decoder_layers=2,
    d_ff=256, dropout=0.1,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(copy_model.parameters(), lr=1e-3)
losses, accuracies = [], []

print("Training copy task (Pre-LN)...")
for epoch in range(n_epochs):
    copy_model.train()
    src, tgt_in, tgt_out = generate_copy_data(batch_size, seq_len, vocab_size)
    src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
    tgt_mask = create_causal_mask(seq_len).to(device)

    logits = copy_model(src, tgt_in, tgt_mask=tgt_mask)
    loss = criterion(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    acc = (logits.argmax(-1) == tgt_out).float().mean().item()
    accuracies.append(acc)
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}: Loss = {loss.item():.4f}, Accuracy = {acc:.2%}")

print(f"\nFinal: Loss = {losses[-1]:.4f}, Accuracy = {accuracies[-1]:.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses)
axes[0].set_title('Copy-task loss')
axes[0].set_xlabel('Step')
axes[1].plot(accuracies)
axes[1].set_title('Copy-task accuracy')
axes[1].set_xlabel('Step')
axes[1].set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

## 9. Autoregressive Generation

At inference time there is no teacher forcing: we generate one token at a time, feeding each prediction back into the decoder.

In [ ]:
def generate(model, src, max_len):
    """Greedy autoregressive generation: one token at a time."""
    model.eval()
    with torch.no_grad():
        encoder_output = model.encode(src)
        generated = torch.ones(1, 1, dtype=torch.long, device=src.device)  # BOS = 1
        for _ in range(max_len):
            tgt_mask = create_causal_mask(generated.size(1)).to(src.device)
            decoder_output = model.decode(generated, encoder_output, tgt_mask=tgt_mask)
            logits = model.output_projection(decoder_output)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            if next_token.item() == 2:  # EOS
                break
    return generated

test_src = torch.randint(3, vocab_size, (1, seq_len)).to(device)
output = generate(copy_model, test_src, max_len=seq_len)

print("Copy Task Test:")
print("-" * 40)
print(f"Input:  {test_src[0].cpu().numpy()}")
print(f"Output: {output[0, 1:].cpu().numpy()}")  # skip BOS
print(f"\nMatch: {torch.equal(test_src[0], output[0, 1:seq_len+1])}")

## 10. Attention Hooks — Peek Inside the Encoder

PyTorch `forward_hook`s let you record the output of a module without touching its code. Here we capture attention weights from each encoder layer after a forward pass.

In [ ]:
copy_model.eval()
test_src = torch.randint(3, vocab_size, (1, 8)).to(device)

attention_weights = []
def hook_fn(module, inputs, output):
    _, weights = output
    attention_weights.append(weights.detach().cpu())

hooks = [layer.attention.register_forward_hook(hook_fn) for layer in copy_model.encoder_layers]
with torch.no_grad():
    _ = copy_model.encode(test_src)
for h in hooks:
    h.remove()

# Visualize the 4 heads from layer 1
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
layer_weights = attention_weights[0][0]  # layer 0, batch 0
for head_idx in range(4):
    ax = axes[head_idx]
    ax.imshow(layer_weights[head_idx].numpy(), cmap='Blues')
    ax.set_title(f'Head {head_idx + 1}')
    ax.set_xlabel('Key Position')
    if head_idx == 0:
        ax.set_ylabel('Query Position')
plt.suptitle(f'Encoder Layer 1 Attention\nInput: {test_src[0].cpu().numpy()}', fontsize=12)
plt.tight_layout()
plt.show()

print("Different heads learn different attention patterns - that's what makes multi-head useful.")

## What Can Go Wrong: Pre-LN vs Post-LN

The 2017 paper placed LayerNorm **after** the residual add: `x = LayerNorm(x + sublayer(x))`. This is called **Post-LN**. It works, but it's notoriously finicky: without a learning-rate warmup schedule, the gradients through the early layers explode and training diverges.

Every modern implementation (GPT-2/3/4, LLaMA, T5, most of HuggingFace) uses **Pre-LN** instead: `x = x + sublayer(LayerNorm(x))`. Normalizing *before* the sub-layer keeps the residual path clean — gradients flow through the identity shortcut unobstructed — and the model trains stably with a vanilla Adam schedule.

Let's prove it. We'll train the same copy task twice with the same optimizer and learning rate, once per variant, and watch the loss curves.

In [ ]:
class PostLNEncoderBlock(nn.Module):
    """Post-LN variant: x = LayerNorm(x + sublayer(x))."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, x, mask=None):
        attn_out, _ = self.attention(x, x, x, mask)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x, None

class PostLNDecoderBlock(nn.Module):
    """Post-LN decoder variant."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm3 = nn.LayerNorm(d_model)
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        self_out, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self_out)
        cross_out, _ = self.cross_attention(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + cross_out)
        ff_out = self.ff(x)
        x = self.norm3(x + ff_out)
        return x, None

def train_variant(enc_block_cls, dec_block_cls, n_steps=80, lr=1e-3):
    m = Transformer(src_vocab_size=vocab_size, tgt_vocab_size=vocab_size,
                    d_model=64, num_heads=4, num_encoder_layers=2,
                    num_decoder_layers=2, d_ff=256, dropout=0.1).to(device)
    # Swap in the variant's blocks
    m.encoder_layers = nn.ModuleList([enc_block_cls(64, 4, 256, 0.1) for _ in range(2)]).to(device)
    m.decoder_layers = nn.ModuleList([dec_block_cls(64, 4, 256, 0.1) for _ in range(2)]).to(device)
    opt = optim.Adam(m.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    curve = []
    for _ in range(n_steps):
        src, tgt_in, tgt_out = generate_copy_data(batch_size, seq_len, vocab_size)
        src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
        tgt_mask = create_causal_mask(seq_len).to(device)
        logits = m(src, tgt_in, tgt_mask=tgt_mask)
        loss = crit(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        curve.append(loss.item())
    return curve

torch.manual_seed(0)
loss_preln = train_variant(TransformerEncoderBlock, TransformerDecoderBlock)
torch.manual_seed(0)
loss_postln = train_variant(PostLNEncoderBlock, PostLNDecoderBlock)

plt.figure(figsize=(8, 4))
plt.plot(loss_preln, label='Pre-LN')
plt.plot(loss_postln, label='Post-LN')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Pre-LN vs Post-LN (no warmup, same lr)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Final loss  Pre-LN: {loss_preln[-1]:.3f}   Post-LN: {loss_postln[-1]:.3f}")
print("Pre-LN drops faster and more smoothly. Post-LN either needs warmup or a much smaller lr to match it.")

## Exercises

In [ ]:
# --- Exercise 1: Scaled Dot-Product Attention from Scratch ---
# This is the core operation inside every transformer layer.
# Given Q, K, V tensors (shape: seq_len x d_k), compute:
#   1. Raw scores = Q @ K.T                      (how much each query matches each key)
#   2. Scaled scores = scores / sqrt(d_k)        (prevent softmax saturation)
#   3. Weights = softmax(scaled, dim=-1)          (normalize to probabilities)
#   4. Output = weights @ V                       (weighted sum of values)

def my_scaled_dot_product_attention(Q, K, V):
    """
    Q: (seq_len, d_k)
    K: (seq_len, d_k)
    V: (seq_len, d_k)
    Returns: output (seq_len, d_k), weights (seq_len, seq_len)
    """
    d_k = Q.shape[-1]

    # YOUR CODE HERE:
    scores = None    # Q @ K.transpose(-2, -1)
    scaled = None    # scores / math.sqrt(d_k)
    weights = None   # F.softmax(scaled, dim=-1)
    output = None    # weights @ V

    return output, weights

# --- Check ---
torch.manual_seed(42)
d_k = 16
Q = torch.randn(5, d_k)
K = torch.randn(5, d_k)
V = torch.randn(5, d_k)

output, weights = my_scaled_dot_product_attention(Q, K, V)
assert output is not None, "Implement the function - don't leave None!"
assert output.shape == (5, d_k), f"Expected output shape (5, {d_k}), got {output.shape}"
assert weights.shape == (5, 5), f"Expected weights shape (5, 5), got {weights.shape}"
assert abs(weights.sum(dim=-1).mean().item() - 1.0) < 1e-5, \
    "Attention weights must sum to 1 per row! (softmax should normalize them)"

# Compare a specific value against the reference implementation
ref_output, ref_weights = scaled_dot_product_attention(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0))
ref_output = ref_output.squeeze(0)
assert torch.allclose(output, ref_output, atol=1e-5), \
    "Your output doesn't match the reference scaled_dot_product_attention - recheck the scaling or matmul order."
print(f"Exercise 1 passed!  (output shape: {output.shape})")
print(f"  Attention weight row sums: {[f'{w:.4f}' for w in weights.sum(dim=-1).tolist()]}  <- all ~1.0")

In [ ]:
# --- Exercise 2: Reverse Task (modify-and-observe) ---
# The Transformer learned to COPY sequences. Now modify the training loop so it learns to REVERSE them.
# Input:  [3, 7, 4, 9, 5]
# Target: [5, 9, 4, 7, 3]
#
# YOUR CODE HERE:
# 1) Change generate_reverse_data() so tgt_output is src reversed (use src.flip(dims=[1]))
# 2) Keep teacher forcing: tgt_input = [BOS, tgt_output[:, :-1]]

def generate_reverse_data(batch_size, seq_len, vocab_size=50):
    sequences = torch.randint(3, vocab_size, (batch_size, seq_len))
    src = sequences
    tgt_output = None  # <- YOUR CODE: reverse src along dim=1
    bos = torch.ones(batch_size, 1, dtype=torch.long)
    tgt_input = None   # <- YOUR CODE: [BOS, tgt_output[:, :-1]]
    return src, tgt_input, tgt_output

# --- Check data generation first ---
_src, _in, _out = generate_reverse_data(2, 5)
assert _out is not None and _in is not None, "Fill in tgt_output and tgt_input!"
assert _out[0].tolist() == _src[0].tolist()[::-1], "tgt_output must be src reversed!"
assert _in[0, 0].item() == 1, "tgt_input must start with BOS=1"
print("Reverse data looks good.")

# --- Train on reverse ---
# NOTE: reverse is harder than copy (the model must learn an anti-diagonal
# cross-attention pattern), so we train for 300 steps instead of 100.
torch.manual_seed(7)
reverse_model = Transformer(
    src_vocab_size=vocab_size, tgt_vocab_size=vocab_size,
    d_model=64, num_heads=4, num_encoder_layers=2,
    num_decoder_layers=2, d_ff=256, dropout=0.1,
).to(device)
rev_opt = optim.Adam(reverse_model.parameters(), lr=1e-3)
rev_crit = nn.CrossEntropyLoss()

for epoch in range(300):
    reverse_model.train()
    src, tgt_in, tgt_out = generate_reverse_data(batch_size, seq_len, vocab_size)
    src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
    tgt_mask = create_causal_mask(seq_len).to(device)
    logits = reverse_model(src, tgt_in, tgt_mask=tgt_mask)
    loss = rev_crit(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))
    rev_opt.zero_grad(); loss.backward(); rev_opt.step()

# --- Evaluate ---
reverse_model.eval()
with torch.no_grad():
    src, tgt_in, tgt_out = generate_reverse_data(128, seq_len, vocab_size)
    src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
    tgt_mask = create_causal_mask(seq_len).to(device)
    preds = reverse_model(src, tgt_in, tgt_mask=tgt_mask).argmax(-1)
    reversed_acc = (preds == tgt_out).float().mean().item()

print(f"Reverse-task token accuracy: {reversed_acc:.1%}")
assert reversed_acc > 0.5, \
    f"Expected reversed_acc > 0.5, got {reversed_acc:.1%} - the model should be able to learn this with Pre-LN + teacher forcing."
print("Exercise 2 passed!")

In [ ]:
# --- Exercise 3: num_heads ablation ---
# You trained TinyClassifier with num_heads=4 earlier and saved its test accuracy as `acc_4head`.
# Now retrain the SAME architecture but with num_heads=1 and store the result in `acc_1head`.
# Assert that more heads helps: acc_1head < acc_4head.

torch.manual_seed(42)
clf_1head = TinyClassifier(num_heads=1)  # <- YOUR CHANGE: one head instead of four
_, _, test_accs_1head = train_tinyclassifier(clf_1head)
acc_1head = test_accs_1head[-1]

print(f"acc_4head = {acc_4head:.1%}")
print(f"acc_1head = {acc_1head:.1%}")
assert acc_1head < acc_4head, \
    f"Expected 1-head accuracy ({acc_1head:.1%}) to be lower than 4-head ({acc_4head:.1%}). " \
    f"If they're equal, the task may be too easy - try a harder task or more seeds."

print("\nWhy multi-head helps:")
print("  With one head, every query attends to every key through a single 32-dim projection.")
print("  With four heads, each head projects into its own 8-dim subspace and attends independently,")
print("  so different heads can specialize in different kinds of relationships (e.g. position-based,")
print("  value-based, long-range vs local). The final W_o mixes those perspectives back together.")
print("\nExercise 3 passed!")

## Summary

**Transformer architecture**:
- **Encoder**: Process input (bidirectional)
- **Decoder**: Generate output (autoregressive)
- **No recurrence**: Fully parallel processing

**Key components**:
1. **Multi-head attention**: Multiple attention perspectives
2. **Feed-forward**: Position-wise transformation
3. **Residual connections**: Enable deep networks
4. **Layer normalization**: Stable training (Pre-LN keeps the residual path clean)
5. **Positional encoding**: Inject position information

**Encoder block (Pre-LN)**:
```
x -> x + Attention(LN(x)) -> x + FFN(LN(x))
```

**Decoder block (Pre-LN)**:
```
x -> x + MaskedAttention(LN(x))
  -> x + CrossAttention(LN(x), encoder_output)
  -> x + FFN(LN(x))
```

**Three variants**:
1. **Encoder-Decoder**: Translation (T5, BART)
2. **Encoder-Only**: Understanding (BERT)
3. **Decoder-Only**: Generation (GPT)

**Why Transformers won**:
- Parallel processing (faster)
- Long-range dependencies (attention)
- Scalable (billions of parameters)
- Transfer learning (pre-train -> fine-tune)

**Next up**: Using pre-trained Transformers with Hugging Face!